# API 인증키 확인

`.env`의 인증키로 외부 API를 한 번씩 호출해 응답을 그대로 본다. 셀마다 독립적으로 실행할 수 있다.

- 출력에서는 인증키 값을 `***`로 가린다.
- `.env`를 고친 뒤에는 **1번 셀부터 다시 실행**해야 새 키가 반영된다(`dotenv_values`는 실행할 때 파일을 읽는다).
- 공공데이터포털 오류 코드: `20` 활용 미승인, `22` 일일 트래픽 초과, `30` 등록되지 않은 서비스키, `31` 기한 만료

In [1]:
import re
from pathlib import Path

import requests
from dotenv import dotenv_values

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists():
  ROOT = ROOT.parent

env = dotenv_values(ROOT / ".env")
SECRETS = [v for v in env.values() if v and len(v) >= 4]


def mask(text: str) -> str:
  for v in SECRETS:
    text = text.replace(v, "***")
  return text


def show(resp: requests.Response, limit: int = 1500) -> None:
  print("HTTP", resp.status_code, "|", resp.headers.get("content-type"))
  print(mask(resp.text[:limit]))
  

for name in ["DATA_GO_KR_KEY", "YOUTHCENTER_KEY", "OC", "QDRANT_URL", "QDRANT_API_KEY"]:
  v = env.get(name) or ""
  print(f"{name:16} {'있음' if v else '없음':3} 길이 {len(v):3}")

DATA_GO_KR_KEY   있음  길이  64
YOUTHCENTER_KEY  있음  길이  36
OC               있음  길이   4
QDRANT_URL       있음  길이  76
QDRANT_API_KEY   있음  길이 176


## 1. 생활법령 백문백답 서비스_GW (REST / XML, `DATA_GO_KR_KEY`)

기존 서비스 폐기 예정 안내에 따라 [신규 서비스_GW](https://www.data.go.kr/data/15159736/openapi.do) 활용신청 및 승인을 완료했다(2026-10-13). 신규 서비스는 **REST / XML**이며 SOAP 본문을 사용하지 않는다.

`https://apis.data.go.kr/1170000/ManyAskManyAnswerService/{operation}`에 HTTPS GET 요청을 보내고, 쿼리의 `serviceKey`에 `.env`의 `DATA_GO_KR_KEY`를 넣는다. `requests`가 인코딩하므로 포털 Encoding 키는 먼저 `unquote`로 디코딩한다. 응답은 UTF-8 XML로 처리한다.

실제 검증: `GET /getMaskMAnswerClassList`가 HTTP 200, `resultCode=00`, `NORMAL SERVICE.`와 분류 목록을 반환했다. 포털 상세기능 표의 분류목록조회/항목조회 설명은 경로명과 뒤바뀌어 표시되어 있으므로 실제 응답을 기준으로 해석한다.

분류 목록의 필드는 기존 SOAP의 camelCase와 달리 소문자(`onhunqnaastnm`, `onhunqnaastseq`, `onhunqueastseq`, `onhunquecatnm`)다. `pageNo`, `numOfRows`, 응답의 `totalCount`로 페이지를 확인한다.


In [2]:
from urllib.parse import unquote
from xml.etree import ElementTree as ET

EASYLAW_URL = "https://apis.data.go.kr/1170000/ManyAskManyAnswerService"


def easylaw_get(operation: str, **params) -> requests.Response:
  # requests가 쿼리 값을 인코딩하므로 포털 Encoding 키도 먼저 디코딩한다.
  key = unquote(env["DATA_GO_KR_KEY"])
  resp = requests.get(
    f"{EASYLAW_URL}/{operation}",
    params={**params, "serviceKey": key},
    timeout=60,
  )
  resp.encoding = "utf-8"
  return resp


resp = easylaw_get("getMaskMAnswerClassList", pageNo=1, numOfRows=1000)
show(resp)
root = ET.fromstring(resp.content)
status = {e.tag: e.text for e in root.iter() if e.tag in {
  "resultCode", "resultMsg", "returnReasonCode", "returnAuthMsg", "errMsg"
}}
print("응답 상태:", status)
if resp.ok and root.findtext("./header/resultCode") == "00":
  classes = sorted({e.text for e in root.iter("onhunqnaastnm") if e.text})
  print("대분류:", len(classes), "개 / 전체 분류:", root.findtext("./body/totalCount"))
  print([c for c in classes if any(w in c for w in ["임대", "주택", "부동산"])] or classes)
else:
  print("→ HTTP 상태와 GW 오류 필드, 활용신청 승인 상태를 확인한다.")


HTTP 200 | application/xml
<?xml version="1.0" encoding="UTF-8" standalone="yes"?><response><header><resultCode>00</resultCode><resultMsg>NORMAL SERVICE.</resultMsg></header><body><items><item><cls>대분류</cls><onhunqnaastnm>가정법률</onhunqnaastnm><onhunqnaastseq>25</onhunqnaastseq><onhunqueastseq>0</onhunqueastseq><onhunquecatnm></onhunquecatnm><seq>1000025</seq><upseq>0</upseq></item><item><cls>중분류</cls><onhunqnaastnm>가정법률</onhunqnaastnm><onhunqnaastseq>25</onhunqnaastseq><onhunqueastseq>1</onhunqueastseq><onhunquecatnm>이혼</onhunquecatnm><seq>1025101</seq><upseq>1000025</upseq></item><item><cls>중분류</cls><onhunqnaastnm>가정법률</onhunqnaastnm><onhunqnaastseq>25</onhunqnaastseq><onhunqueastseq>2</onhunqueastseq><onhunquecatnm>재혼</onhunquecatnm><seq>1025102</seq><upseq>1000025</upseq></item><item><cls>중분류</cls><onhunqnaastnm>가정법률</onhunqnaastnm><onhunqnaastseq>25</onhunqnaastseq><onhunqueastseq>5</onhunqueastseq><onhunquecatnm>유언</onhunquecatnm><seq>1025105</seq><upseq>1000025</upseq></item><item

## 2. 복지로 중앙부처 복지서비스 (REST, `DATA_GO_KR_KEY`)

같은 키로 다른 API를 불러 **키 자체가 유효한지** 대조한다. 여기서 정상이면 키는 맞고, 1번 실패는 해당 서비스 쪽 등록 문제다.

In [3]:
resp = requests.get(
  "https://apis.data.go.kr/B554287/NationalWelfareInformationsV001/NationalWelfarelistV001",
  params={"serviceKey": env["DATA_GO_KR_KEY"], "callTp": "L", "pageNo": 1, "numOfRows": 3, "srchKeyCode": "001", "searchWrd": "주거"},
  timeout=60,
)
show(resp)

HTTP 200 | application/xml;charset=utf-8
<?xml version="1.0" encoding="UTF-8" standalone="yes"?><wantedList><totalCount>7</totalCount><pageNo>1</pageNo><numOfRows>3</numOfRows><resultCode>0</resultCode><resultMessage>SUCCESS</resultMessage><servList><inqNum>50151</inqNum><intrsThemaArray>생활지원,주거</intrsThemaArray><jurMnofNm>국토교통부</jurMnofNm><jurOrgNm>주거복지정책과</jurOrgNm><onapPsbltYn>N</onapPsbltYn><rprsCtadr>1599-0001</rprsCtadr><servDgst>쪽방·고시원 등 열악한 비주택거주자의 공공임대주택 이주수요를 발굴하고 LH와 협력하여 이주과정을 지원합니다.</servDgst><servDtlLink>https://www.bokjiro.go.kr/ssis-tbu/twataa/wlfareInfo/moveTWAT52011M.do?wlfareInfoId=WLF00000036&amp;wlfareInfoReldBztpCd=01</servDtlLink><servId>WLF00000036</servId><servNm>주거취약계층 주거상향 지원사업</servNm><sprtCycNm>1회성</sprtCycNm><srvPvsnNm>기타</srvPvsnNm><svcfrstRegTs>20210903</svcfrstRegTs><trgterIndvdlArray>저소득</trgterIndvdlArray></servList><servList><inqNum>88729</inqNum><intrsThemaArray>주거</intrsThemaArray><jurMnofNm>보건복지부</jurMnofNm><jurOrgNm>기초생활보장과</jurOrgNm><onapPsbltYn

## 3. 온통청년 청년정책 (`YOUTHCENTER_KEY`)

In [4]:
resp = requests.get(
  "https://www.youthcenter.go.kr/go/ythip/getPlcy",
  params={"apiKeyNm": env["YOUTHCENTER_KEY"], "pageNum": 1, "pageSize": 2, "rtnType": "json", "plcyNm": "월세"},
  timeout=60,
)
show(resp, limit=800)

HTTP 200 | application/json
{"resultCode":200,"resultMessage":"성공적으로 데이터를 가지고 왔습니다.","result":{"pagging":{"totCount":64,"pageNum":1,"pageSize":2},"youthPolicyList":[{"plcyNo":"20261004005400113874","bscPlanCycl":"2","bscPlanPlcyWayNo":"003","bscPlanFcsAsmtNo":"009","bscPlanAsmtNo":"030","pvsnInstGroupCd":"0054001","plcyPvsnMthdCd":"0042006","plcyAprvSttsCd":"0044002","plcyNm":"청년월세 한시 특별지원","plcyKywdNm":"보조금,주거지원","plcyExplnCn":"소득기준에 따라 월 200천원 임차료 지원(최대 24회)\n※ 실제 납부하는 월세(임대료)의 최대 20만원까지 지원","lclsfNm":"주거","mclsfNm":"전월세 및 주거급여 지원","plcySprtCn":"소득기준에 따라 월 200천원 임차료 지원(최대 24회)\n※ 실제 납부하는 월세(임대료)의 최대 20만원까지 지원","sprvsnInstCd":"1613000","sprvsnInstCdNm":"국토교통부","sprvsnInstPicNm":"","operInstCd":"       ","operInstCdNm":"","operInstPicNm":"","sprtSclLmtYn":"N","aplyPrdSeCd":"0057001","bizPrdSeCd":"0056002","bizPrdBgng


## 4. 법령 API (`OC`)

In [5]:
resp = requests.get(
  "https://www.law.go.kr/DRF/lawSearch.do",
  params={"OC": env["OC"], "target": "law", "type": "JSON", "query": "주택임대차보호법", "display": 1},
  timeout=60,
)
show(resp, limit=800)

HTTP 200 | application/json;charset=UTF-8
{"LawSearch": {
    "law": {
        "현행연혁코드": "현행",
        "법령일련번호": "276291",
        "자법타법여부": "",
        "법령상세링크": "/DRF/lawService.do?OC=***&target=law&MST=276291&type=HTML&mobileYn=&efYd=20260102",
        "법령명한글": "주택임대차보호법",
        "법령구분명": "법률",
        "소관부처명": "국토교통부,법무부",
        "공포번호": "21065",
        "제개정구분명": "타법개정",
        "소관부처코드": "1270000,1613000",
        "id": "1",
        "법령ID": "001248",
        "공동부령정보": "",
        "시행일자": "20260102",
        "공포일자": "20251001",
        "법령약칭명": "주택임대차법"
    },
    "resultMsg": "success",
    "키워드": "주택임대차보호법",
    "page": "1",
    "resultCode": "00",
    "target": "law",
    "totalCnt": "2",
    "section": "lawNm",
    "numOfRows": "1"
}}


## 5. Qdrant (`QDRANT_URL`, `QDRANT_API_KEY`)

In [6]:
resp = requests.get(env["QDRANT_URL"].rstrip("/") + "/collections", headers={"api-key": env["QDRANT_API_KEY"]}, timeout=30)
show(resp)

HTTP 200 | application/json
{"result":{"collections":[]},"status":"ok","time":4.217e-6}
